# Sensitivity, robustness and loop shaping: notebook

The running example of the Note: the robot's heading loop. Yaw inertia I = 0.1 kg·m², torque limit 0.2 N·m, PD gains kp = 0.4 N·m/rad and kd = 0.08 N·m·s/rad (RO-061's first try). We check every number in the Note: the sensitivity functions, the robustness test, the lead, lag and lead-lag designs, the waterbed integral and the time responses. No randomness is used.

In [1]:
import numpy as np
from scipy.integrate import quad
from scipy.optimize import brentq, fsolve

I, TAU_MAX = 0.1, 0.2            # yaw inertia (kg m^2), torque limit (N m)
KP, KD = 0.4, 0.08               # PD heading gains
DRAG = 0.02                      # one wheel drags 0.2 N more, 0.1 m from the centre (N m)
I_PAY, TAU_REAL = 0.15, 0.25     # the real robot: payload inertia, motor-driver delay (s)
K, A, B = 1.875, 2.5 / 3, 7.5    # lead k (s + a)/(s + b)
A2, B2 = 0.25, 0.025             # lag (s + a2)/(s + b2)

P = lambda s, inertia=I: 1 / (inertia * s**2)
C_pd = lambda s, kp=KP, kd=KD: kp + kd * s
C_lead = lambda s: K * (s + A) / (s + B)
C_ll = lambda s: C_lead(s) * (s + A2) / (s + B2)

def margins(L, lo=1e-3, hi=1e3):
    wgc = brentq(lambda x: abs(L(1j * x)) - 1, lo, hi)
    pm = (np.degrees(np.angle(L(1j * wgc))) + 360) % 360 - 180
    return wgc, pm, np.radians(pm) / wgc

def sens_peak(L, lo=1e-3, hi=1e3):
    w = np.logspace(np.log10(lo), np.log10(hi), 800001)
    d = np.abs(1 + L(1j * w))
    return w[d.argmin()], d.min(), 1 / d.min()

L_pd = lambda s: C_pd(s) * P(s)
print("model ready")

model ready


## 1. The opening problem (Note, Section 1)

Steady state under a constant drag: the turn rate is zero, so the PD torque kp θ balances the drag.

In [2]:
print("PD offset:", DRAG / KP, "rad =", round(np.degrees(DRAG / KP), 2), "deg")

def sim(kind, t, dist=DRAG, t_dist=1.0, inertia=I, tau=0.0, b_lag=B2):
    dt, n = t[1] - t[0], len(t)
    lag = int(round(tau / dt))
    th, om = np.zeros(n), np.zeros(n)
    x1 = x2 = 0.0
    d = np.where(t >= t_dist, dist, 0.0)
    tq = np.zeros(n)
    for i in range(n - 1):
        j = max(i - lag, 0)
        e = -th[j]
        if kind == "pd":
            u = KP * e - KD * om[j]
        else:
            u1 = K * e + K * (A - B) * x1
            x1 += dt * (-B * x1 + e)
            u = u1 if kind == "lead" else u1 + (A2 - b_lag) * x2
            x2 += dt * (-b_lag * x2 + u1)
        u = np.clip(u, -TAU_MAX, TAU_MAX)
        tq[i] = u
        om[i + 1] = om[i] + dt * (u + d[i]) / inertia
        th[i + 1] = th[i] + dt * om[i + 1]
    return th, tq

t = np.arange(0, 30.001, 1e-3)
th, _ = sim("pd", t)
print("PD as modelled: peak", th.max().round(4), "final", th[-1].round(4))
th, tq = sim("pd", t, inertia=I_PAY, tau=TAU_REAL)
print("PD, real robot: max by 20 s", th[t <= 20].max().round(3), " first torque limit at", t[np.argmax(np.abs(tq) >= TAU_MAX - 1e-12)], "s")
f = lambda x: [(I_PAY * complex(*x)**2 + (0.08 * complex(*x) + 0.4) * np.exp(-complex(*x) * TAU_REAL)).real,
               (I_PAY * complex(*x)**2 + (0.08 * complex(*x) + 0.4) * np.exp(-complex(*x) * TAU_REAL)).imag]
print("PD, real robot: growing closed-loop pole pair", fsolve(f, [0, 1.7]).round(3))

PD offset: 0.049999999999999996 rad = 2.86 deg


PD as modelled: peak 0.0763 final 0.05


PD, real robot: max by 20 s 0.256  first torque limit at 28.908 s
PD, real robot: growing closed-loop pole pair [0.084 1.67 ]


## 2. The sensitivity function from the Nyquist plot (Note, Section 2)

In [3]:
w0, d0, Ms = sens_peak(L_pd)
print("closest approach at", round(w0, 3), "rad/s: L =", np.round(L_pd(1j * w0), 3), " distance", round(d0, 4), " Ms", round(Ms, 3))
print("margins (w_gc, PM, delay margin):", np.round(margins(L_pd), 4))
print("check: 2 sin(PM/2) =", round(2 * np.sin(np.radians(margins(L_pd)[1]) / 2), 4))
print("L(j0.5) =", L_pd(0.5j), " |1 + L| =", round(abs(1 + L_pd(0.5j)), 3), " |S| =", round(1 / abs(1 + L_pd(0.5j)), 4))

closest approach at 2.085 rad/s: L = (-0.92-0.384j)  distance 0.3919  Ms 2.552
margins (w_gc, PM, delay margin): [ 2.0815 22.6023  0.1895]
check: 2 sin(PM/2) = 0.3919
L(j0.5) = (-16-1.6j)  |1 + L| = 15.085  |S| = 0.0663


## 3. Disturbances and noise (Note, Sections 3 and 4)

In [4]:
S = lambda s: 1 / (1 + L_pd(s)); T = lambda s: L_pd(s) / (1 + L_pd(s))
wsc = brentq(lambda x: abs(S(1j * x)) - 1, 0.1, 2)
w = np.logspace(-1, 2, 200001)
print("|S| = 1 at", round(wsc, 3), "rad/s;  Mt =", round(np.abs(T(1j * w)).max(), 3), "at", round(w[np.abs(T(1j * w)).argmax()], 3))
print("S + T = 1 everywhere:", np.allclose(S(1j * w) + T(1j * w), 1))
for x in (0.5, 2.085, 10, 100):
    print(f"w = {x}: |S| = {abs(S(1j*x)):.4f}, |T| = {abs(T(1j*x)):.4f}, L = {L_pd(1j*x):.4f}")

|S| = 1 at 1.474 rad/s;  Mt = 2.734 at 1.929
S + T = 1 everywhere: True
w = 0.5: |S| = 0.0663, |T| = 1.0659, L = -16.0000-1.6000j
w = 2.085: |S| = 2.5516, |T| = 2.5437, L = -0.9201-0.3837j
w = 10: |S| = 1.0381, |T| = 0.0928, L = -0.0400-0.0800j
w = 100: |S| = 1.0004, |T| = 0.0080, L = -0.0004-0.0080j


## 4. The Gang of Four (Note, Section 5)

In [5]:
def gang(Cf, w, inertia=I):
    s = 1j * w; L = Cf(s) * P(s, inertia); S = 1 / (1 + L)
    return dict(S=S, T=L * S, PS=P(s, inertia) * S, CS=Cf(s) * S)
g = gang(C_pd, np.array([1e-4, 2.085, 100.0]))
for k, v in g.items():
    print(k, np.round(np.abs(v), 4))

S [0.     2.5516 1.0004]
T [1.     2.5437 0.008 ]
PS [2.5000e+00 5.8694e+00 1.0000e-03]
CS [0.     1.1058 8.0129]


## 5. Robustness to the real robot's model error (Note, Section 6)

In [6]:
delta = lambda w: (I / I_PAY) * np.exp(-1j * w * TAU_REAL) - 1
w = np.logspace(np.log10(0.3), 1, 20001)
for name, Cf in (("PD", C_pd), ("PD kd 0.4", lambda s: C_pd(s, KP, 0.4)), ("lead-lag", C_ll)):
    Tw = gang(Cf, w)["T"]
    bad = np.abs(delta(w)) > 1 / np.abs(Tw)
    print(name, " max |delta| |T| =", round((np.abs(delta(w)) * np.abs(Tw)).max(), 3),
          " fails between", np.round(w[bad][[0, -1]], 2) if bad.any() else "nowhere")
for x in (1.4, 1.7, 2.0, 2.4, 3.0):
    c = L_pd(1j * x)
    print(f"disk at {x}: centre {c:.3f}, radius {abs(c*delta(x)):.3f}, distance to -1 {abs(c+1):.3f}")
print("w = 2: |delta| =", round(abs(delta(2.0)), 3), " 1/|T| =", round(1/abs(T(2j)), 3))
for name, Cf in (("PD", C_pd), ("lead-lag", C_ll)):
    Lr = lambda s: Cf(s) * np.exp(-s * TAU_REAL) / (I_PAY * s**2)
    print(name, "on the real robot: PM", round(margins(Lr)[1], 1), "deg")

PD  max |delta| |T| = 1.413  fails between [1.59 2.4 ]
PD kd 0.4  max |delta| |T| = 0.712  fails between nowhere
lead-lag  max |delta| |T| = 0.743  fails between nowhere
disk at 1.4: centre -2.041-0.571j, radius 0.929, distance to -1 1.187
disk at 1.7: centre -1.384-0.471j, radius 0.701, distance to -1 0.607
disk at 2.0: centre -1.000-0.400j, radius 0.564, distance to -1 0.400
disk at 2.4: centre -0.694-0.333j, radius 0.452, distance to -1 0.452
disk at 3.0: centre -0.444-0.267j, radius 0.355, distance to -1 0.616
w = 2: |delta| = 0.524  1/|T| = 0.371
PD on the real robot: PM -5.5 deg
lead-lag on the real robot: PM 18.8 deg


## 6. More derivative: Ms against noise (Note, Section 7.2)

In [7]:
for kd in (0.04, 0.06, 0.08, 0.12, 0.16, 0.2, 0.3, 0.4):
    L = lambda s: C_pd(s, KP, kd) * P(s)
    _, d, Ms = sens_peak(L)
    d, Ms = (1.0, 1.0) if Ms < 1 else (d, Ms)
    print(f"kd {kd}: zeta {kd/(2*np.sqrt(KP*I)):.2f}, closest {d:.3f}, Ms {Ms:.2f}, PM {margins(L)[1]:.1f}, noise |CS(100j)| {abs(gang(lambda s: C_pd(s, KP, kd), np.array([100.0]))['CS'][0]):.1f}")

kd 0.04: zeta 0.10, closest 0.199, Ms 5.03, PM 11.4, noise |CS(100j)| 4.0


kd 0.06: zeta 0.15, closest 0.297, Ms 3.37, PM 17.1, noise |CS(100j)| 6.0


kd 0.08: zeta 0.20, closest 0.392, Ms 2.55, PM 22.6, noise |CS(100j)| 8.0


kd 0.12: zeta 0.30, closest 0.572, Ms 1.75, PM 33.3, noise |CS(100j)| 12.0


kd 0.16: zeta 0.40, closest 0.733, Ms 1.36, PM 43.1, noise |CS(100j)| 16.0


kd 0.2: zeta 0.50, closest 0.866, Ms 1.15, PM 51.8, noise |CS(100j)| 20.0


kd 0.3: zeta 0.75, closest 1.000, Ms 1.00, PM 67.7, noise |CS(100j)| 30.0
kd 0.4: zeta 1.00, closest 1.000, Ms 1.00, PM 76.3, noise |CS(100j)| 40.0


## 7. The lead compensator (Note, Section 7.3)

In [8]:
print("max phase where w^2 = a b:", A * B, "-> w =", np.sqrt(A * B))
print("phase lead at 2.5 rad/s:", round(np.degrees(np.arctan(2.5 / A) - np.arctan(2.5 / B)), 2), "deg")
print("|C/k| at 2.5:", round(abs((2.5j + A) / (2.5j + B)), 4), " |P(j2.5)|:", abs(P(2.5j)), " k =", round(1 / (abs((2.5j + A) / (2.5j + B)) * abs(P(2.5j))), 4))
for r in (1.5, 2, 3, 4.5, 6, 9):
    a, b = 2.5 / np.sqrt(r), 2.5 * np.sqrt(r)
    k = 1 / (abs((2.5j + a) / (2.5j + b)) * abs(P(2.5j)))
    L = lambda s: k * (s + a) / (s + b) * P(s)
    print(f"b/a {r}: k {k:.3f}, zero {a:.3f}, pole {b:.3f}, PM {margins(L)[1]:.1f}, Ms {sens_peak(L)[2]:.2f}, |C(100j)| {abs(k*(100j+a)/(100j+b)):.2f}, C(0) {k*a/b:.3f}")
L_lead = lambda s: C_lead(s) * P(s)
print("lead closed-loop poles:", np.round(np.roots(np.polyadd(np.polymul([I, 0, 0], [1, B]), K * np.array([1, A]))), 3))
print("plain gain 0.625: L(j2.5) =", 0.625 * P(2.5j))

max phase where w^2 = a b: 6.25 -> w = 2.5
phase lead at 2.5 rad/s: 53.13 deg
|C/k| at 2.5: 0.3333  |P(j2.5)|: 1.6  k = 1.875


b/a 1.5: k 0.765, zero 2.041, pole 3.062, PM 11.5, Ms 5.00, |C(100j)| 0.77, C(0) 0.510


b/a 2: k 0.884, zero 1.768, pole 3.536, PM 19.5, Ms 3.00, |C(100j)| 0.88, C(0) 0.442


b/a 3: k 1.083, zero 1.443, pole 4.330, PM 30.0, Ms 2.01, |C(100j)| 1.08, C(0) 0.361


b/a 4.5: k 1.326, zero 1.179, pole 5.303, PM 39.5, Ms 1.59, |C(100j)| 1.32, C(0) 0.295


b/a 6: k 1.531, zero 1.021, pole 6.124, PM 45.6, Ms 1.43, |C(100j)| 1.53, C(0) 0.255
b/a 9: k 1.875, zero 0.833, pole 7.500, PM 53.1, Ms 1.30, |C(100j)| 1.87, C(0) 0.208
lead closed-loop poles: [-2.5+0.j -2.5+0.j -2.5-0.j]
plain gain 0.625: L(j2.5) = (-1-0j)


## 8. The lag and the lead-lag (Note, Sections 7.4 and 7.5)

In [9]:
for r in (1, 2, 4, 6, 10, np.inf):
    b2 = 0.0 if np.isinf(r) else A2 / r
    Cf = lambda s: C_lead(s) * (s + A2) / (s + b2)
    L = lambda s: Cf(s) * P(s)
    off = 0.0 if b2 == 0 else DRAG / abs(Cf(1e-9j))
    print(f"lag ratio {r}: PM {margins(L)[1]:.1f}, Ms {sens_peak(L)[2]:.3f}, drag offset {off:.4f} rad")
L_ll = lambda s: C_ll(s) * P(s)
wgc, pm, dm = margins(L_ll); wms, d, Ms = sens_peak(L_ll)
print("lead-lag: w_gc", round(wgc, 3), " PM", round(pm, 2), " delay margin", round(dm, 3), " Ms", round(Ms, 3), " closest", round(d, 3))
print("C(0) =", round(abs(C_ll(1e-9j)), 4), " offset", round(DRAG / abs(C_ll(1e-9j)), 4), " |C(100j)| =", round(abs(C_ll(100j)), 3), " PD |C(100j)| =", round(abs(C_pd(100j)), 3))
print("phase of the lag at 2.5 rad/s:", round(np.degrees(np.arctan(2.5 / A2) - np.arctan(2.5 / B2)), 2), "deg")
p = np.polyadd(np.polymul([I, 0, 0], np.polymul([1, B], [1, B2])), K * np.polymul([1, A], [1, A2]))
print("lead-lag closed-loop poles:", np.round(np.roots(p), 3))
for kind in ("lead", "leadlag"):
    th, tq = sim(kind, t)
    print(kind, "as modelled: peak", th.max().round(4), " at 10 s", th[t.searchsorted(10)].round(4), " final", th[-1].round(4), " max torque", np.abs(tq).max().round(3))
    th, tq = sim(kind, t, inertia=I_PAY, tau=TAU_REAL)
    print(kind, "real robot: peak", th.max().round(4), " swing over the last 5 s", np.ptp(th[t > 25]).round(5))

lag ratio 1: PM 53.1, Ms 1.299, drag offset 0.0960 rad
lag ratio 2: PM 50.3, Ms 1.326, drag offset 0.0480 rad


lag ratio 4: PM 48.9, Ms 1.342, drag offset 0.0240 rad


lag ratio 6: PM 48.4, Ms 1.347, drag offset 0.0160 rad
lag ratio 10: PM 48.0, Ms 1.352, drag offset 0.0096 rad


lag ratio inf: PM 47.4, Ms 1.359, drag offset 0.0000 rad


lead-lag: w_gc 2.51  PM 48.01  delay margin 0.334  Ms 1.352  closest 0.74
C(0) = 2.0833  offset 0.0096  |C(100j)| = 1.87  PD |C(100j)| = 8.01
phase of the lag at 2.5 rad/s: -5.14 deg
lead-lag closed-loop poles: [-3.998+0.j    -1.643+1.159j -1.643-1.159j -0.242+0.j   ]


lead as modelled: peak 0.096  at 10 s 0.096  final 0.096  max torque 0.025


lead real robot: peak 0.1074  swing over the last 5 s 0.0


leadlag as modelled: peak 0.0712  at 10 s 0.0215  final 0.0097  max torque 0.027


leadlag real robot: peak 0.0949  swing over the last 5 s 0.00021


## 8b. RO-060's tuned PD (Note, Sections 5, 6.2, 7.2 and 7.5)

RO-060 tuned the heading loop to kp = 0.2 N·m/rad and kd = 0.283 N·m·s/rad (plus ki = 0.05 against the dragging gearbox, left out here).

In [10]:
C_tuned = lambda s: C_pd(s, 0.2, 0.283)
L_tuned = lambda s: C_tuned(s) * P(s)
wgc, pm, dm = margins(L_tuned)
w = np.logspace(-3, 3, 800001)
print("tuned PD: w_gc", round(wgc, 3), " PM", round(pm, 2), " delay margin", round(dm, 3), " max |S|", round(np.abs(1 / (1 + L_tuned(1j * w))).max(), 4))
print("noise gain at 100 rad/s:", round(abs(gang(C_tuned, np.array([100.0]))['CS'][0]), 2), " drag offset:", DRAG / 0.2, "rad")
print("noise ratios: tuned/ours", round(28.3 / 8.0, 2), " tuned/lead-lag", round(28.3 / 1.87, 1))
Lr = lambda s: C_tuned(s) * np.exp(-s * TAU_REAL) / (I_PAY * s**2)
wr = np.logspace(-3, 3, 800001); ph = np.unwrap(np.angle(Lr(1j * wr))); i = np.where(np.diff(np.sign(ph + np.pi)))[0][0]
print("real robot: w_gc, PM", np.round(margins(Lr)[:2], 2), " gain margin", round(1 / abs(Lr(1j * wr[i])), 2), " Ms", round(sens_peak(Lr)[2], 2))
wd = np.logspace(-2, 1, 20001)
print("robust test, largest |delta| |T|:", round((np.abs(delta(wd)) * np.abs(gang(C_tuned, wd)['T'])).max(), 3))

tuned PD: w_gc 2.912  PM 76.36  delay margin 0.458  max |S| 1.0
noise gain at 100 rad/s: 28.3  drag offset: 0.09999999999999999 rad
noise ratios: tuned/ours 3.54  tuned/lead-lag 15.1


real robot: w_gc, PM [ 2.   41.89]  gain margin 3.05  Ms 1.73
robust test, largest |delta| |T|: 0.544


## 9. The waterbed (Note, Section 8)

In [11]:
for name, Cf in (("lead", C_lead), ("lead-lag", C_ll)):
    f = lambda x: np.log(abs(1 / (1 + Cf(1j * x) * P(1j * x))))
    wsc = brentq(f, 0.5, 5)
    neg = quad(f, 0, wsc, limit=500, points=[0.025, 0.25, 0.833])[0]
    pos = sum(quad(f, lo, hi, limit=500)[0] for lo, hi in ((wsc, 15), (15, 1e3), (1e3, 1e6)))
    print(f"{name}: |S| = 1 at {wsc:.3f} rad/s, area below {neg:.3f}, above {pos:.3f}, sum {neg+pos:.1e}, above beyond 15 rad/s {pos - quad(f, wsc, 15, limit=500)[0]:.3f}")

lead: |S| = 1 at 2.134 rad/s, area below -3.220, above 3.220, sum -1.9e-05, above beyond 15 rad/s 1.182
lead-lag: |S| = 1 at 1.899 rad/s, area below -3.449, above 3.449, sum -1.9e-05, above beyond 15 rad/s 1.184


## 10. Apollo's lead-lag (Note, Section 9)

`leadlag_controller.cc` builds C(s) = β(τ s + 1)/(α τ s + 1) and discretises it with the bilinear (Tustin) rule. The test configuration `leadlag_speed_conf.pb.txt` sets α = 0.35, β = 1, τ = 0.1; the control period is ts = 0.01 s.

In [12]:
al, be, ta, Ts = 0.35, 1.0, 0.1, 0.01
zero, pole = 1 / ta, 1 / (al * ta)
wm = np.sqrt(zero * pole)
print("zero", zero, "rad/s, pole", round(pole, 2), "rad/s, peak lead", round(np.degrees(np.arctan(wm / zero) - np.arctan(wm / pole)), 1), "deg at", round(wm, 1), "rad/s")
a1, a0, b1, b0 = al * ta, 1.0, be * ta, be
kn1, kn0, kd1, kd0 = 2 * b1 + Ts * b0, Ts * b0 - 2 * b1, 2 * a1 + Ts * a0, Ts * a0 - 2 * a1
z = np.exp(1j * wm * Ts)
Hd = (kn1 + kn0 / z) / (kd1 + kd0 / z); Hc = (b1 * 1j * wm + b0) / (a1 * 1j * wm + a0)
print("at the peak: discrete phase", round(np.degrees(np.angle(Hd)), 2), "deg, continuous", round(np.degrees(np.angle(Hc)), 2), "deg")

zero 10.0 rad/s, pole 28.57 rad/s, peak lead 28.8 deg at 16.9 rad/s
at the peak: discrete phase 28.78 deg, continuous 28.78 deg
